In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from typing import TypedDict
import subprocess
from openai import OpenAI
import textwrap
from langchain.chat_models import init_chat_model
from typing_extensions import Annotated
import operator
import base64

llm = init_chat_model("openai:gpt-4o-mini")


class State(TypedDict):

    video_file: str
    audio_file: str
    transcription: str
    summaries: Annotated[list[str], operator.add]
    thumbnail_prompts: Annotated[list[str], operator.add]
    thumbnail_sketches: Annotated[list[str], operator.add]
    final_summary: str

In [ ]:
def extract_audio(state: State):
    output_file = state["video_file"].replace("mp4", "mp3")
    command = [
        "ffmpeg",
        "-i",
        state["video_file"],
        "-filter:a",
        "atempo=2.0",
        "-y",
        output_file,
    ]
    subprocess.run(command)
    return {
        "audio_file": output_file,
    }


def transcribe_audio(state: State):
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=audio_file,
            language="ko",
            prompt="",
        )
        return {
            "transcription": transcription,
        }

def dispatch_summarizers(state: State):
    transcription = state["transcription"]
    chunks = []
    for i, chunk in enumerate(textwrap.wrap(transcription, 500)):
        chunks.append({"id": i + 1, "chunk": chunk})
    return [Send("summarize_chunk", chunk) for chunk in chunks]


def summarize_chunk(chunk):
    chunk_id = chunk["id"]
    chunk = chunk["chunk"]

    response = llm.invoke(
        f"""
        Please summarize the following text.

        Text: {chunk}
        """
    )
    summary = f"[Chunk {chunk_id}] {response.content}"
    return {
        "summaries": [summary],
    }


def mega_summary(state: State):

    all_summaries = "\n".join(state["summaries"])

    prompt = f"""
        You are given multiple summaries of different chunks from a video transcription.

        Please create a comprehensive final summary that combines all the key points.

        Individual summaries:

        {all_summaries}
    """

    response = llm.invoke(prompt)

    return {
        "final_summary": response.content,
    }


def dispatch_artists(state: State):
    return [
        Send(
            "generate_thumbnails",
            {
                "id": i,
                "summary": state["final_summary"],
            },
        )
        for i in [1, 2, 3, 4, 5]
    ]


def generate_thumbnails(args):
    concept_id = args["id"]
    summary = args["summary"]

    prompt = f"""
    Based on this video summary, create a detailed visual prompt for a YouTube thumbnail.

    Create a detailed prompt for generating a thumbnail image that would attract viewers. Include:
        - Main visual elements
        - Color scheme
        - Text overlay suggestions
        - Overall composition
    
    Summary: {summary}
    """

    response = llm.invoke(prompt)

    thumbnail_prompt = response.content

    client = OpenAI()

    result = client.images.generate(
        model="gpt-image-1",
        prompt=thumbnail_prompt,
        quality="low",
        moderation="low",
        size="auto",
    )

    image_bytes = base64.b64decode(result.data[0].b64_json)

    filename = f"thumbnail_{concept_id}.jpg"

    with open(filename, "wb") as file:
        file.write(image_bytes)

    return {
        "thumbnail_prompts": [thumbnail_prompt],
        "thumbnail_sketches": [filename],
    }

In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_node("summarize_chunk", summarize_chunk)
graph_builder.add_node("mega_summary", mega_summary)
graph_builder.add_node("generate_thumbnails", generate_thumbnails)

graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_conditional_edges(
    "transcribe_audio", dispatch_summarizers, ["summarize_chunk"]
)
graph_builder.add_edge("summarize_chunk", "mega_summary")
graph_builder.add_conditional_edges(
    "mega_summary", dispatch_artists, ["generate_thumbnails"]
)
graph_builder.add_edge("generate_thumbnails", END)

graph = graph_builder.compile()

In [8]:
graph.invoke({"video_file": "video.mp4"})

{'video_file': 'video.mp4',
 'audio_file': 'video.mp3',
 'transcription': 'SJM은 뉴딩킹, 라이트 액션이라는 슬로건 아래 모든 경영활동에서 새로운 생각, 새로운 시도를 통해 기업 가치를 높이고 있습니다. SJM은 대한민국 최초의 밸러우즈 국산화 제조기업으로 자동차, 플랜트, 건축, 조선해역, 뉴모빌리티 분야에서 혁신제품을 잇따라 선보이며 국내 점유율 1위, 세계 점유율 2위의 글로벌 기술전문기업으로 독보적인 위상을 쌓아가고 있습니다. SJM의 밸러우즈 기술은 자동차 분야에서는 배기계와 엔진의 진동 및 열평창 흡수를 위해 적용되며, 플랜트 분야의 발전플랜트 및 석유화학, 절단, 가스자료, 건축 분야의 공기조화, 내지지 및 수방개통, 조선해역 분야에서는 LNG캐리어, LNG범퍼링, 선박엔진 등에 적용되며, 전기차, 수소차, 우주항공, 초고속운송체계, 핵융합발전 등 뉴모빌리티 및 하이테크 분야까지 끊임없이 도전하고 있습니다. 1975년 성진기공을 창업, 1996년 SJM으로 사명을 변경하였습니다. 이후 지속적으로 해외 글로벌 거점을 마련하면서 성장과 발전을 거듭 하였고, 경쟁력 있는 글로벌 리더로서 시장을 선도하고 있으며, 친환경 시대에 발맞춰 뉴모빌리티를 포함 다양한 분야로 사업 영역을 확대 하고 있습니다. SJM은 한국을 기반으로 고객에게 최대의 가치를 제공하기 위해 아시아 에는 중국, 일본, 말레이시아, 미주 지역에는 미국, 멕시코, 유럽 및 아프리카에는 독일, 모로코, 남아프리카 공화국의 글로벌 네트워크를 구축 하였습니다. 8개의 생산 거점을 통해 공급 기반을 갖췄으며, 7개의 영업 거점을 통해 고객사와 탄탄한 파트너십을 유지하고 있습니다. 또한 7개의 전문화된 R&D 기술 지원 거점을 운영하여 신속하게 고객 요구에 대응하고 있습니다. 지금 이 순간에도 SJM은 고객과 소통 하며 글로벌 기업으로 성장하고 있습니다. SJM의 제품은 독자적인 연구 개발과 완벽품질에 기반합니다. 차량용 벨로우즈 설계, 